# HSTU_CANONICAL_v1 — finalization notebook

This notebook **does not train HSTU**. It loads the best ML-1M core and
large checkpoints already stored on Google Drive and finishes the canonical
baseline:

1. quality checkpoint validation;
2. exact cached one-event inference;
3. full-history ↔ cached numerical parity;
4. Triton-fused cached HSTU attention;
5. cached PyTorch ↔ Triton numerical parity;
6. end-to-end cached serving latency and minimal K/V cache memory;
7. writes `HSTU_CANONICAL_v1_stamp.json` to Drive.

The quality architecture and weights are frozen.

In [ ]:
# Colab environment check.
import torch, sys
print("torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Switch Colab runtime to GPU."

In [ ]:
# Triton normally ships with CUDA PyTorch in Colab.
# Install only if the import is genuinely absent; no TorchRec/FBGEMM needed.
try:
    import triton, triton.language as tl
    print("Triton:", triton.__version__)
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton, triton.language as tl
    print("Triton installed:", triton.__version__)

## Timestamp parity note

Meta's public ML-1M feature path exposes the **next event timestamp** to
HSTU's relative-time bias while holding out the next item ID.

Therefore the exact parity tests below call:

```python
cached_step(..., current_timestamp=t_i, next_timestamp=t_{i+1})
```

For real online serving, `t_{i+1}` is not known. The cached API therefore
also supports `next_timestamp=None`, which falls back to the current
timestamp. We keep these two semantics explicitly separated.

In [ ]:
# ============================================================
# HSTU_CANONICAL_v1 FINALIZER — COLAB
#
# Purpose
# -------
# Freeze the already-reproduced HSTU quality model and finish:
#   1) exact cached incremental inference;
#   2) full-history <-> cached parity;
#   3) Triton-fused cached attention;
#   4) cached PyTorch <-> Triton parity;
#   5) serving latency + cache-memory measurements;
#   6) write a machine-readable stamp to Google Drive.
#
# This file DOES NOT TRAIN HSTU.

In [ ]:
# ============================================================

import os, sys, math, json, time, random, subprocess, importlib
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Use a GPU Colab runtime."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# -----------------------------
# Colab / Drive
# -----------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/hstu_pure_pytorch_ml1m")
DATA_ROOT = Path("/content/hstu_ml1m_data")
STAMP_PATH = DRIVE_ROOT / "HSTU_CANONICAL_v1_stamp.json"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount note:", repr(e))

DATA_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Controls
# -----------------------------

VARIANTS = ("core", "large")
SEED = 42

QUALITY_TARGETS = {
    "core": {"NDCG@10": 0.1720, "HR@10": 0.3097},
    "large": {"NDCG@10": 0.1893, "HR@10": 0.3294},
}
QUALITY_TOL_NDCG = 0.005

PARITY_ATOL = 5e-5
TRITON_PARITY_ATOL = 5e-4

# End-to-end cached latency uses the canonical n=211 model.
BENCH_HISTORY = (16, 50, 100, 200)
BENCH_BATCHES = (1, 32, 128)
BENCH_WARMUP = 50
BENCH_ITERS = 300

In [ ]:
# ============================================================
# DATA PREPARATION

In [ ]:
# ============================================================

ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

def prepare_ml1m():
    import urllib.request, zipfile as pyzip

    zip_path = DATA_ROOT / "ml-1m.zip"
    ratings_path = DATA_ROOT / "ml-1m" / "ratings.dat"
    movies_path = DATA_ROOT / "ml-1m" / "movies.dat"

    if not zip_path.exists():
        print("Downloading:", ML1M_URL)
        urllib.request.urlretrieve(ML1M_URL, zip_path)
    if not ratings_path.exists():
        with pyzip.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_ROOT)

    ratings = pd.read_csv(
        ratings_path,
        sep="::",
        engine="python",
        names=["user_id", "movie_id", "rating", "timestamp"],
    )
    movies = pd.read_csv(
        movies_path,
        sep="::",
        engine="python",
        names=["movie_id", "title", "genres"],
        encoding="latin-1",
    )

    ratings = ratings.sort_values("timestamp")
    seqs = []
    for uid, g in ratings.groupby("user_id", sort=True):
        seqs.append(
            (
                int(uid),
                g["movie_id"].astype(np.int64).to_numpy(),
                g["timestamp"].astype(np.int64).to_numpy(),
                g["rating"].astype(np.int64).to_numpy(),
            )
        )

    catalog_ids = np.sort(movies["movie_id"].unique()).astype(np.int64)
    max_item_id = int(max(catalog_ids.max(), ratings["movie_id"].max()))
    unique_rated = int(ratings["movie_id"].nunique())

    stats = {
        "users": len(seqs),
        "ratings": int(len(ratings)),
        "unique_rated_items": unique_rated,
        "catalog_items": int(len(catalog_ids)),
        "max_item_id": max_item_id,
        "avg_sequence_len": float(np.mean([len(x[1]) for x in seqs])),
    }
    assert unique_rated == 3706
    assert max_item_id == 3952
    print("DATA:", stats)
    return seqs, catalog_ids, max_item_id, stats

In [ ]:
# ============================================================
# INITIALIZATION + CANONICAL MODEL

In [ ]:
# ============================================================

def truncated_normal_(x: torch.Tensor, mean: float, std: float):
    # Same algorithm used in the quality reproduction.
    with torch.no_grad():
        size = x.shape
        tmp = x.new_empty(size + (4,)).normal_()
        valid = (tmp < 2) & (tmp > -2)
        ind = valid.max(-1, keepdim=True)[1]
        x.copy_(tmp.gather(-1, ind).squeeze(-1))
        x.mul_(std).add_(mean)
    return x


class RelativeBucketedTimeAndPositionBias(nn.Module):
    def __init__(self, n: int, num_buckets: int = 128):
        super().__init__()
        self.n = n
        self.num_buckets = num_buckets
        self.ts_w = nn.Parameter(torch.empty(num_buckets + 1))
        self.pos_w = nn.Parameter(torch.empty(2 * n - 1))
        nn.init.normal_(self.ts_w, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_w, mean=0.0, std=0.02)

        i = torch.arange(n)[:, None]
        j = torch.arange(n)[None, :]
        rel_pos_idx = j - i + (n - 1)
        self.register_buffer("rel_pos_idx", rel_pos_idx.long(), persistent=False)

    def forward_from_buckets(self, time_buckets):
        return (
            self.pos_w[self.rel_pos_idx].unsqueeze(0)
            + self.ts_w[time_buckets]
        )


class HSTUBlock(nn.Module):
    def __init__(self, d_model, heads, dqk, dv, dropout, n):
        super().__init__()
        self.d_model = d_model
        self.heads = heads
        self.dqk = dqk
        self.dv = dv
        self.dropout = dropout
        self.n = n
        self.eps = 1e-6

        total = 2 * dv * heads + 2 * dqk * heads
        self.uvqk = nn.Parameter(torch.empty(d_model, total))
        nn.init.normal_(self.uvqk, mean=0.0, std=0.02)

        self.rel_bias = RelativeBucketedTimeAndPositionBias(n=n, num_buckets=128)

        self.o = nn.Linear(dv * heads, d_model)
        nn.init.xavier_uniform_(self.o.weight)

        causal = torch.tril(torch.ones(n, n, dtype=torch.float32))
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, time_buckets, lengths):
        B, N, D = x.shape
        valid = (
            torch.arange(N, device=x.device)[None, :] < lengths[:, None]
        ).unsqueeze(-1)
        x = x * valid.to(x.dtype)

        normed = F.layer_norm(x, [D], eps=self.eps)
        z = F.silu(normed @ self.uvqk)
        split = [
            self.dv * self.heads,
            self.dv * self.heads,
            self.dqk * self.heads,
            self.dqk * self.heads,
        ]
        u, v, q, k = torch.split(z, split, dim=-1)

        valid_f = valid.to(z.dtype)
        u, v, q, k = [a * valid_f for a in (u, v, q, k)]

        q = q.view(B, N, self.heads, self.dqk).permute(0, 2, 1, 3).contiguous()
        k = k.view(B, N, self.heads, self.dqk).permute(0, 2, 1, 3).contiguous()
        v = v.view(B, N, self.heads, self.dv).permute(0, 2, 1, 3).contiguous()

        qk = q @ k.transpose(-2, -1)
        qk = qk + self.rel_bias.forward_from_buckets(time_buckets).unsqueeze(1)
        qk = F.silu(qk) / N
        qk = qk * self.causal.view(1, 1, N, N)

        attn = (qk @ v).permute(0, 2, 1, 3).contiguous()
        attn = attn.reshape(B, N, self.heads * self.dv)
        attn = F.layer_norm(attn, [self.heads * self.dv], eps=self.eps)

        u = u.view(B, N, self.heads * self.dv)
        y = self.o(F.dropout(u * attn, p=self.dropout, training=self.training))
        y = y + x
        return y * valid.to(y.dtype)


class HSTU(nn.Module):
    def __init__(
        self,
        max_item_id,
        d_model,
        n,
        layers,
        heads,
        dqk,
        dv,
        dropout,
        l2_eps=1e-6,
    ):
        super().__init__()
        self.max_item_id = max_item_id
        self.d_model = d_model
        self.n = n
        self.l2_eps = l2_eps
        self.layers = layers
        self.heads = heads
        self.dqk = dqk
        self.dv = dv

        self.item = nn.Embedding(max_item_id + 1, d_model, padding_idx=0)
        truncated_normal_(self.item.weight, 0.0, 0.02)
        with torch.no_grad():
            self.item.weight[0].zero_()

        self.pos = nn.Embedding(n, d_model)
        truncated_normal_(self.pos.weight, 0.0, math.sqrt(1.0 / d_model))
        self.input_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            HSTUBlock(d_model, heads, dqk, dv, dropout, n)
            for _ in range(layers)
        ])

    def encode_all(self, ids, timestamps, lengths):
        B, N = ids.shape
        pos = torch.arange(N, device=ids.device)[None, :].expand(B, N)

        x = self.item(ids) * math.sqrt(self.d_model) + self.pos(pos)
        x = self.input_dropout(x)
        x = x * (ids != 0).unsqueeze(-1).to(x.dtype)

        ext_ts = torch.cat([timestamps, timestamps[:, N - 1 : N]], dim=1)
        delta = ext_ts[:, 1:].unsqueeze(2) - ext_ts[:, :-1].unsqueeze(1)
        buckets = (
            torch.log(torch.abs(delta).clamp(min=1).float()) / 0.301
        ).long().clamp(0, 128).detach()

        for block in self.blocks:
            x = block(x, buckets, lengths)

        return F.normalize(x.float(), p=2, dim=-1, eps=self.l2_eps).to(x.dtype)

    def current_embedding(self, ids, timestamps, lengths):
        x = self.encode_all(ids, timestamps, lengths)
        row = torch.arange(ids.size(0), device=ids.device)
        return x[row, lengths - 1]


def geometry(variant):
    if variant == "core":
        return dict(d_model=50, layers=2, heads=1, dqk=50, dv=50)
    if variant == "large":
        return dict(d_model=50, layers=8, heads=2, dqk=25, dv=25)
    raise ValueError(variant)


def checkpoint_path(variant):
    return DRIVE_ROOT / f"hstu_{variant}_seed{SEED}" / "best.pt"


def load_canonical_model(variant, max_item_id):
    path = checkpoint_path(variant)
    assert path.exists(), f"Missing checkpoint: {path}"
    ck = torch.load(path, map_location=DEVICE, weights_only=False)

    cfg = ck["config"]
    n = int(cfg["max_len"] + cfg["max_output_len"])
    g = geometry(variant)

    model = HSTU(
        max_item_id=max_item_id,
        n=n,
        dropout=float(cfg["dropout"]),
        l2_eps=float(cfg.get("l2_eps", 1e-6)),
        **g,
    ).to(DEVICE)
    model.load_state_dict(ck["model"])
    model.eval()

    metrics = ck.get("metrics", {})
    print("CHECKPOINT", variant, path, "epoch", ck.get("epoch"), metrics)
    return model, ck

In [ ]:
# ============================================================
# EXACT CACHED INCREMENTAL INFERENCE — PURE PYTORCH

In [ ]:
# ============================================================

@dataclass
class StreamingCache:
    k: list
    v: list
    timestamps: torch.Tensor
    lengths: torch.Tensor


def make_streaming_cache(model, batch_size, dtype=torch.float32):
    k = [
        torch.zeros(
            batch_size, block.heads, model.n, block.dqk,
            device=DEVICE, dtype=dtype,
        )
        for block in model.blocks
    ]
    v = [
        torch.zeros(
            batch_size, block.heads, model.n, block.dv,
            device=DEVICE, dtype=dtype,
        )
        for block in model.blocks
    ]
    timestamps = torch.zeros(
        batch_size, model.n, device=DEVICE, dtype=torch.long
    )
    lengths = torch.zeros(batch_size, device=DEVICE, dtype=torch.long)
    return StreamingCache(k=k, v=v, timestamps=timestamps, lengths=lengths)


def _scatter_at_positions(cache_tensor, values, pos):
    # cache_tensor [B,H,N,D], values [B,H,D], pos [B]
    B, H, _, D = cache_tensor.shape
    idx = pos[:, None, None, None].expand(B, H, 1, D)
    cache_tensor.scatter_(2, idx, values.unsqueeze(2))


def _scatter_ts(timestamps, values, pos):
    timestamps.scatter_(1, pos[:, None], values[:, None])


def cached_attention_torch(block, q, k_cache, v_cache, timestamps, pos, next_timestamp):
    # q [B,H,DQ], caches [B,H,N,D*]
    B, H, N, _ = k_cache.shape
    key_pos = torch.arange(N, device=q.device)[None, :]
    valid = key_pos <= pos[:, None]

    scores = torch.einsum("bhd,bhnd->bhn", q, k_cache)

    rel_idx = key_pos - pos[:, None] + (N - 1)
    rel_idx = rel_idx.clamp(0, 2 * N - 2)
    pos_bias = block.rel_bias.pos_w[rel_idx]  # [B,N]

    delta = next_timestamp[:, None] - timestamps
    buckets = (
        torch.log(torch.abs(delta).clamp(min=1).float()) / 0.301
    ).long().clamp(0, 128)
    time_bias = block.rel_bias.ts_w[buckets]

    scores = scores + (pos_bias + time_bias).unsqueeze(1)
    weights = F.silu(scores) / N
    weights = weights * valid[:, None, :].to(weights.dtype)

    return torch.einsum("bhn,bhnd->bhd", weights, v_cache)


@torch.inference_mode()
def cached_step_torch(
    model,
    cache,
    item_ids,
    current_timestamp,
    next_timestamp=None,
    advance=True,
):
    """
    Append one observed item and return the canonical HSTU state for that item.

    Exact-reproduction mode:
        next_timestamp = timestamp of the event being predicted next.

    Real online serving fallback:
        next_timestamp = current_timestamp
    because the future timestamp is not known.
    """
    model.eval()
    B = item_ids.shape[0]
    pos = cache.lengths.clone()
    assert int(pos.max()) < model.n

    if next_timestamp is None:
        next_timestamp = current_timestamp

    _scatter_ts(cache.timestamps, current_timestamp, pos)

    x = (
        model.item(item_ids) * math.sqrt(model.d_model)
        + model.pos(pos)
    )

    for li, block in enumerate(model.blocks):
        normed = F.layer_norm(x, [model.d_model], eps=block.eps)
        z = F.silu(normed @ block.uvqk)

        split = [
            block.dv * block.heads,
            block.dv * block.heads,
            block.dqk * block.heads,
            block.dqk * block.heads,
        ]
        u, v_new, q_new, k_new = torch.split(z, split, dim=-1)

        u = u.view(B, block.heads, block.dv)
        v_new = v_new.view(B, block.heads, block.dv)
        q_new = q_new.view(B, block.heads, block.dqk)
        k_new = k_new.view(B, block.heads, block.dqk)

        _scatter_at_positions(cache.k[li], k_new, pos)
        _scatter_at_positions(cache.v[li], v_new, pos)

        attn = cached_attention_torch(
            block=block,
            q=q_new,
            k_cache=cache.k[li],
            v_cache=cache.v[li],
            timestamps=cache.timestamps,
            pos=pos,
            next_timestamp=next_timestamp,
        )

        attn = attn.reshape(B, block.heads * block.dv)
        attn = F.layer_norm(
            attn,
            [block.heads * block.dv],
            eps=block.eps,
        )
        x = block.o(u.reshape(B, -1) * attn) + x

    out = F.normalize(x.float(), p=2, dim=-1, eps=model.l2_eps).to(x.dtype)
    if advance:
        cache.lengths.add_(1)
    return out

In [ ]:
# ============================================================
# FULL-HISTORY <-> CACHED PARITY

In [ ]:
# ============================================================

def make_full_prefix_tensors(ids_np, ts_np, prefix_len, model_n):
    ids = torch.zeros(1, model_n, device=DEVICE, dtype=torch.long)
    ts = torch.zeros(1, model_n, device=DEVICE, dtype=torch.long)

    ids[0, :prefix_len] = torch.as_tensor(
        ids_np[:prefix_len], device=DEVICE, dtype=torch.long
    )
    ts[0, :prefix_len + 1] = torch.as_tensor(
        ts_np[:prefix_len + 1], device=DEVICE, dtype=torch.long
    )
    lengths = torch.tensor([prefix_len], device=DEVICE)
    return ids, ts, lengths


@torch.inference_mode()
def parity_one_window(model, ids_np, ts_np, checkpoints=(1, 2, 5, 10, 25, 50, 100, 200)):
    # Need one future timestamp after every tested prefix.
    usable = min(len(ids_np) - 1, model.n - 1)
    ids_np = ids_np[:usable + 1]
    ts_np = ts_np[:usable + 1]

    cache = make_streaming_cache(model, batch_size=1)
    errors = []

    wanted = set(x for x in checkpoints if x <= usable)

    for j in range(usable):
        item = torch.tensor([int(ids_np[j])], device=DEVICE)
        cur_ts = torch.tensor([int(ts_np[j])], device=DEVICE)
        nxt_ts = torch.tensor([int(ts_np[j + 1])], device=DEVICE)

        cached = cached_step_torch(
            model, cache, item, cur_ts, next_timestamp=nxt_ts, advance=True
        )

        prefix_len = j + 1
        if prefix_len in wanted:
            ids, ts, lengths = make_full_prefix_tensors(
                ids_np, ts_np, prefix_len, model.n
            )
            full = model.current_embedding(ids, ts, lengths)

            max_abs = float((cached - full).abs().max().cpu())
            cos = float(F.cosine_similarity(cached.float(), full.float()).item())
            errors.append({
                "prefix": prefix_len,
                "max_abs": max_abs,
                "cosine": cos,
            })

    return errors


def run_full_cached_parity(model, seqs, num_users=3):
    # Use suffix windows to match ML-1M truncation semantics for long histories.
    chosen = [0, len(seqs) // 3, 2 * len(seqs) // 3][:num_users]
    all_rows = []

    for ui in chosen:
        _, ids, ts, _ = seqs[ui]
        window = min(model.n, len(ids))
        ids_w = ids[-window:]
        ts_w = ts[-window:]

        rows = parity_one_window(model, ids_w, ts_w)
        for r in rows:
            r["user_index"] = ui
            all_rows.append(r)

    worst = max((r["max_abs"] for r in all_rows), default=0.0)
    min_cos = min((r["cosine"] for r in all_rows), default=1.0)
    print("FULL↔CACHED parity:", {"worst_max_abs": worst, "min_cosine": min_cos})
    return all_rows, worst, min_cos

In [ ]:
# ============================================================
# TRITON FUSED CACHED ATTENTION

In [ ]:
# ============================================================

TRITON_AVAILABLE = False
TRITON_IMPORT_ERROR = None

try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
except Exception as e:
    TRITON_IMPORT_ERROR = repr(e)


if TRITON_AVAILABLE:

    @triton.jit
    def _hstu_cached_attention_kernel(
        Q, K, V,
        KEY_TS, NEXT_TS,
        POS_W, TS_W,
        LENGTHS,
        OUT,
        stride_qb: tl.constexpr,
        stride_qh: tl.constexpr,
        stride_qd: tl.constexpr,
        stride_kb: tl.constexpr,
        stride_kh: tl.constexpr,
        stride_kn: tl.constexpr,
        stride_kd: tl.constexpr,
        stride_vb: tl.constexpr,
        stride_vh: tl.constexpr,
        stride_vn: tl.constexpr,
        stride_vd: tl.constexpr,
        stride_tsb: tl.constexpr,
        stride_tsn: tl.constexpr,
        stride_ob: tl.constexpr,
        stride_oh: tl.constexpr,
        stride_od: tl.constexpr,
        H: tl.constexpr,
        N_MODEL: tl.constexpr,
        DQ: tl.constexpr,
        DV: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_DQ: tl.constexpr,
        BLOCK_DV: tl.constexpr,
    ):
        pid = tl.program_id(0)
        b = pid // H
        h = pid - b * H

        length = tl.load(LENGTHS + b).to(tl.int32)
        pos = length - 1

        d_q = tl.arange(0, BLOCK_DQ)
        q = tl.load(
            Q + b * stride_qb + h * stride_qh + d_q * stride_qd,
            mask=d_q < DQ,
            other=0.0,
        ).to(tl.float32)

        d_v = tl.arange(0, BLOCK_DV)
        acc = tl.zeros([BLOCK_DV], dtype=tl.float32)

        next_ts = tl.load(NEXT_TS + b).to(tl.float32)

        for start in range(0, N_MODEL, BLOCK_N):
            n = start + tl.arange(0, BLOCK_N)
            valid_n = n < length

            k_ptrs = (
                K
                + b * stride_kb
                + h * stride_kh
                + n[:, None] * stride_kn
                + d_q[None, :] * stride_kd
            )
            k = tl.load(
                k_ptrs,
                mask=valid_n[:, None] & (d_q[None, :] < DQ),
                other=0.0,
            ).to(tl.float32)

            score = tl.sum(k * q[None, :], axis=1)

            rel_idx = n - pos + (N_MODEL - 1)
            pos_bias = tl.load(
                POS_W + rel_idx,
                mask=valid_n,
                other=0.0,
            ).to(tl.float32)

            key_ts = tl.load(
                KEY_TS + b * stride_tsb + n * stride_tsn,
                mask=valid_n,
                other=0,
            ).to(tl.float32)
            delta = tl.abs(next_ts - key_ts)
            delta = tl.maximum(delta, 1.0)
            bucket = (tl.log(delta) / 0.301).to(tl.int32)
            bucket = tl.minimum(tl.maximum(bucket, 0), 128)
            time_bias = tl.load(
                TS_W + bucket,
                mask=valid_n,
                other=0.0,
            ).to(tl.float32)

            z = score + pos_bias + time_bias
            w = (z * tl.sigmoid(z)) / N_MODEL
            w = tl.where(valid_n, w, 0.0)

            v_ptrs = (
                V
                + b * stride_vb
                + h * stride_vh
                + n[:, None] * stride_vn
                + d_v[None, :] * stride_vd
            )
            vv = tl.load(
                v_ptrs,
                mask=valid_n[:, None] & (d_v[None, :] < DV),
                other=0.0,
            ).to(tl.float32)

            acc += tl.sum(w[:, None] * vv, axis=0)

        tl.store(
            OUT + b * stride_ob + h * stride_oh + d_v * stride_od,
            acc,
            mask=d_v < DV,
        )


def cached_attention_triton(block, q, k_cache, v_cache, timestamps, lengths, next_timestamp):
    if not TRITON_AVAILABLE:
        raise RuntimeError(f"Triton unavailable: {TRITON_IMPORT_ERROR}")

    B, H, N, _ = k_cache.shape
    out = torch.empty(
        B, H, block.dv, device=q.device, dtype=torch.float32
    )

    # lengths here include the newly inserted current item.
    grid = (B * H,)

    block_dq = 1
    while block_dq < block.dqk:
        block_dq *= 2
    block_dv = 1
    while block_dv < block.dv:
        block_dv *= 2

    _hstu_cached_attention_kernel[grid](
        q, k_cache, v_cache,
        timestamps, next_timestamp,
        block.rel_bias.pos_w,
        block.rel_bias.ts_w,
        lengths,
        out,
        q.stride(0), q.stride(1), q.stride(2),
        k_cache.stride(0), k_cache.stride(1), k_cache.stride(2), k_cache.stride(3),
        v_cache.stride(0), v_cache.stride(1), v_cache.stride(2), v_cache.stride(3),
        timestamps.stride(0), timestamps.stride(1),
        out.stride(0), out.stride(1), out.stride(2),
        H=H,
        N_MODEL=N,
        DQ=block.dqk,
        DV=block.dv,
        BLOCK_N=32,
        BLOCK_DQ=block_dq,
        BLOCK_DV=block_dv,
        num_warps=4,
    )
    return out.to(q.dtype)


@torch.inference_mode()
def cached_step_triton(
    model,
    cache,
    item_ids,
    current_timestamp,
    next_timestamp=None,
    advance=True,
):
    if next_timestamp is None:
        next_timestamp = current_timestamp

    B = item_ids.shape[0]
    pos = cache.lengths.clone()
    assert int(pos.max()) < model.n

    _scatter_ts(cache.timestamps, current_timestamp, pos)

    x = model.item(item_ids) * math.sqrt(model.d_model) + model.pos(pos)

    for li, block in enumerate(model.blocks):
        normed = F.layer_norm(x, [model.d_model], eps=block.eps)
        z = F.silu(normed @ block.uvqk)

        split = [
            block.dv * block.heads,
            block.dv * block.heads,
            block.dqk * block.heads,
            block.dqk * block.heads,
        ]
        u, v_new, q_new, k_new = torch.split(z, split, dim=-1)

        u = u.view(B, block.heads, block.dv)
        v_new = v_new.view(B, block.heads, block.dv)
        q_new = q_new.view(B, block.heads, block.dqk)
        k_new = k_new.view(B, block.heads, block.dqk)

        _scatter_at_positions(cache.k[li], k_new, pos)
        _scatter_at_positions(cache.v[li], v_new, pos)

        # Triton kernel expects length including the just-inserted position.
        lengths_including_current = pos + 1
        attn = cached_attention_triton(
            block,
            q_new,
            cache.k[li],
            cache.v[li],
            cache.timestamps,
            lengths_including_current,
            next_timestamp,
        )

        attn = attn.reshape(B, block.heads * block.dv)
        attn = F.layer_norm(attn, [block.heads * block.dv], eps=block.eps)
        x = block.o(u.reshape(B, -1) * attn) + x

    out = F.normalize(x.float(), p=2, dim=-1, eps=model.l2_eps).to(x.dtype)
    if advance:
        cache.lengths.add_(1)
    return out

In [ ]:
# ============================================================
# CACHED PYTORCH <-> TRITON PARITY

In [ ]:
# ============================================================

@torch.inference_mode()
def compare_torch_triton_one_window(model, ids_np, ts_np, max_steps=100):
    if not TRITON_AVAILABLE:
        return [], float("inf"), 0.0

    usable = min(len(ids_np) - 1, model.n - 1, max_steps)
    ids_np = ids_np[:usable + 1]
    ts_np = ts_np[:usable + 1]

    c_torch = make_streaming_cache(model, 1)
    c_tri = make_streaming_cache(model, 1)

    rows = []
    for j in range(usable):
        item = torch.tensor([int(ids_np[j])], device=DEVICE)
        cur = torch.tensor([int(ts_np[j])], device=DEVICE)
        nxt = torch.tensor([int(ts_np[j + 1])], device=DEVICE)

        a = cached_step_torch(model, c_torch, item, cur, nxt, advance=True)
        b = cached_step_triton(model, c_tri, item, cur, nxt, advance=True)

        if j + 1 in (1, 2, 5, 10, 25, 50, 100):
            err = float((a - b).abs().max().cpu())
            cos = float(F.cosine_similarity(a.float(), b.float()).item())
            rows.append({"prefix": j + 1, "max_abs": err, "cosine": cos})

    worst = max((r["max_abs"] for r in rows), default=0.0)
    min_cos = min((r["cosine"] for r in rows), default=1.0)
    print("CACHED Torch↔Triton:", {"worst_max_abs": worst, "min_cosine": min_cos})
    return rows, worst, min_cos

In [ ]:
# ============================================================
# LATENCY BENCHMARKS

In [ ]:
# ============================================================

def cuda_bench(fn, warmup=50, iters=300):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    times = []
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    for _ in range(iters):
        start.record()
        fn()
        end.record()
        end.synchronize()
        times.append(start.elapsed_time(end))

    arr = np.asarray(times)
    return {
        "mean_ms": float(arr.mean()),
        "p50_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "p99_ms": float(np.percentile(arr, 99)),
    }


def seed_cache_for_benchmark(model, batch_size, history_len):
    assert history_len < model.n
    cache = make_streaming_cache(model, batch_size)

    # Timing only: K/V values need not come from a real history.
    for li, block in enumerate(model.blocks):
        cache.k[li].normal_(0, 0.2)
        cache.v[li].normal_(0, 0.2)

    base = 1_600_000_000
    steps = torch.arange(model.n, device=DEVICE, dtype=torch.long) * 3600
    cache.timestamps[:] = base + steps[None, :]
    cache.lengths.fill_(history_len)

    item_ids = torch.randint(
        1, model.max_item_id + 1,
        (batch_size,),
        device=DEVICE,
        dtype=torch.long,
    )
    current_ts = torch.full(
        (batch_size,),
        base + history_len * 3600,
        device=DEVICE,
        dtype=torch.long,
    )
    next_ts = current_ts + 3600
    return cache, item_ids, current_ts, next_ts


@torch.inference_mode()
def benchmark_cached_model(model, variant):
    rows = []
    for B in BENCH_BATCHES:
        for L in BENCH_HISTORY:
            if L >= model.n:
                continue

            c_pt, item, cur, nxt = seed_cache_for_benchmark(model, B, L)

            # advance=False keeps the cache at a fixed history length.
            pt = cuda_bench(
                lambda: cached_step_torch(
                    model, c_pt, item, cur, nxt, advance=False
                ),
                BENCH_WARMUP, BENCH_ITERS,
            )
            rows.append({
                "variant": variant,
                "backend": "cached_pytorch",
                "batch": B,
                "history": L,
                **pt,
            })

            if TRITON_AVAILABLE:
                c_tri, item2, cur2, nxt2 = seed_cache_for_benchmark(model, B, L)
                tri = cuda_bench(
                    lambda: cached_step_triton(
                        model, c_tri, item2, cur2, nxt2, advance=False
                    ),
                    BENCH_WARMUP, BENCH_ITERS,
                )
                rows.append({
                    "variant": variant,
                    "backend": "cached_triton_attention",
                    "batch": B,
                    "history": L,
                    **tri,
                })

    return rows


def cache_bytes_per_request(model, history_len, dtype_bytes=4):
    total = 0
    for block in model.blocks:
        total += (
            block.heads * history_len * (block.dqk + block.dv) * dtype_bytes
        )
    # timestamps: int64
    total += history_len * 8
    return int(total)

In [ ]:
# ============================================================
# STAMP

In [ ]:
# ============================================================

def quality_pass(variant, ck):
    ndcg = float(ck.get("metrics", {}).get("NDCG@10", float("nan")))
    target = QUALITY_TARGETS[variant]["NDCG@10"]
    return abs(ndcg - target) <= QUALITY_TOL_NDCG, ndcg, target


def finalize():
    seqs, catalog_ids, max_item_id, data_stats = prepare_ml1m()

    stamp = {
        "name": "HSTU_CANONICAL_v1",
        "device": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "triton_available": TRITON_AVAILABLE,
        "data": data_stats,
        "variants": {},
        "latency": [],
        "status": "NOT_STAMPED",
        "notes": {
            "timestamp_semantics": (
                "Exact public-reproduction parity uses the next event timestamp "
                "in HSTU relative-time bias. Real online serving must choose a "
                "policy because that future timestamp is unavailable."
            )
        },
    }

    overall = True

    for variant in VARIANTS:
        print("\n" + "=" * 90)
        print("FINALIZING", variant)
        print("=" * 90)

        model, ck = load_canonical_model(variant, max_item_id)

        qp, qval, qtarget = quality_pass(variant, ck)
        overall &= qp

        full_rows, full_worst, full_cos = run_full_cached_parity(
            model, seqs, num_users=3
        )
        cached_pass = full_worst <= PARITY_ATOL
        overall &= cached_pass

        # Use a real suffix window for Triton parity.
        _, ids, ts, _ = seqs[len(seqs) // 2]
        w = min(len(ids), model.n)
        ids_w, ts_w = ids[-w:], ts[-w:]

        tri_rows, tri_worst, tri_cos = compare_torch_triton_one_window(
            model, ids_w, ts_w, max_steps=min(100, model.n - 1)
        )
        triton_pass = (
            TRITON_AVAILABLE and tri_worst <= TRITON_PARITY_ATOL
        )
        overall &= triton_pass

        bench = benchmark_cached_model(model, variant)
        stamp["latency"].extend(bench)

        stamp["variants"][variant] = {
            "checkpoint": str(checkpoint_path(variant)),
            "checkpoint_epoch": int(ck.get("epoch", -1)),
            "checkpoint_metrics": ck.get("metrics", {}),
            "quality_target": QUALITY_TARGETS[variant],
            "quality_pass": qp,
            "full_cached_parity": {
                "pass": cached_pass,
                "worst_max_abs": full_worst,
                "min_cosine": full_cos,
                "rows": full_rows,
            },
            "torch_triton_parity": {
                "pass": triton_pass,
                "worst_max_abs": tri_worst,
                "min_cosine": tri_cos,
                "rows": tri_rows,
            },
            "minimal_kv_cache_bytes": {
                str(L): cache_bytes_per_request(model, L)
                for L in (50, 200)
                if L < model.n
            },
        }

        print({
            "variant": variant,
            "quality": qval,
            "target": qtarget,
            "quality_pass": qp,
            "full_cached_worst": full_worst,
            "cached_pass": cached_pass,
            "triton_worst": tri_worst,
            "triton_pass": triton_pass,
        })

    stamp["status"] = "STAMPED" if overall else "NOT_STAMPED"
    STAMP_PATH.write_text(json.dumps(stamp, indent=2))
    print("\nSTAMP:", stamp["status"])
    print("Saved:", STAMP_PATH)

    # Compact summary table.
    rows = []
    for v, info in stamp["variants"].items():
        rows.append({
            "variant": v,
            "NDCG@10": info["checkpoint_metrics"].get("NDCG@10"),
            "quality_pass": info["quality_pass"],
            "full_cached_max_abs": info["full_cached_parity"]["worst_max_abs"],
            "full_cached_pass": info["full_cached_parity"]["pass"],
            "triton_max_abs": info["torch_triton_parity"]["worst_max_abs"],
            "triton_pass": info["torch_triton_parity"]["pass"],
        })
    return stamp, pd.DataFrame(rows), pd.DataFrame(stamp["latency"])

In [ ]:
# Run the complete stamp suite.
stamp, summary_df, latency_df = finalize()
display(summary_df)
display(latency_df)

In [ ]:
# Convenient latency pivots.
if len(latency_df):
    display(
        latency_df.pivot_table(
            index=["variant", "batch", "history"],
            columns="backend",
            values="p50_ms",
        ).reset_index()
    )

In [ ]:
# Final stamp status.
print("HSTU_CANONICAL_v1:", stamp["status"])
print("Stamp file:", "/content/drive/MyDrive/hstu_pure_pytorch_ml1m/HSTU_CANONICAL_v1_stamp.json")